# WTI COT Mapping: CFTC + ICE → Marouen's cot_db

Goal: show exactly how to reconstruct the CL rows in `cot_db.csv` from public data.

In [1]:
import pandas as pd

# Load source files
disagg = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)
legacy = pd.read_csv("../downloads/cftc/legacy_combined.csv", low_memory=False)
marouen = pd.read_csv("../cot_data/cot_db.csv")

print(f"disaggregated_combined: {len(disagg)} rows")
print(f"legacy_combined:       {len(legacy)} rows")
print(f"marouen cot_db:        {len(marouen)} rows")

disaggregated_combined: 184173 rows
legacy_combined:       269179 rows
marouen cot_db:        19776 rows


In [2]:
disagg['report_date_as_yyyy_mm_dd'].min(), disagg['report_date_as_yyyy_mm_dd'].max()

('2006-06-13T00:00:00.000', '2026-03-24T00:00:00.000')

In [3]:
legacy['report_date_as_yyyy_mm_dd'].min(), legacy['report_date_as_yyyy_mm_dd'].max()

('1995-03-21T00:00:00.000', '2026-03-24T00:00:00.000')

## Step 1 — Filter CFTC disaggregated for WTI contracts

CL in cot_db = NYMEX WTI (`067651`) + ICE Brent (`067411`), from the **combined** (futures + options) report.

In [4]:
disagg.columns.tolist()

['id',
 'market_and_exchange_names',
 'report_date_as_yyyy_mm_dd',
 'yyyy_report_week_ww',
 'contract_market_name',
 'cftc_contract_market_code',
 'cftc_market_code',
 'cftc_region_code',
 'cftc_commodity_code',
 'commodity_name',
 'open_interest_all',
 'prod_merc_positions_long',
 'prod_merc_positions_short',
 'swap_positions_long_all',
 'swap__positions_short_all',
 'swap__positions_spread_all',
 'm_money_positions_long_all',
 'm_money_positions_short_all',
 'm_money_positions_spread',
 'other_rept_positions_long',
 'other_rept_positions_short',
 'other_rept_positions_spread',
 'tot_rept_positions_long_all',
 'tot_rept_positions_short',
 'nonrept_positions_long_all',
 'nonrept_positions_short_all',
 'open_interest_old',
 'prod_merc_positions_long_1',
 'prod_merc_positions_short_1',
 'swap_positions_long_old',
 'swap__positions_short_old',
 'swap__positions_spread_old',
 'm_money_positions_long_old',
 'm_money_positions_short_old',
 'm_money_positions_spread_1',
 'other_rept_positions

In [5]:
# What do these contracts look like in the file?
for code in ["067651", "067411"]:
    names = disagg[disagg["cftc_contract_market_code"] == code]["market_and_exchange_names"].unique()
    commodity_names = disagg[disagg["cftc_contract_market_code"] == code]["commodity_name"].unique()
    contract_market_names = disagg[disagg["cftc_contract_market_code"] == code]["contract_market_name"].unique()

    
    print(f"\nCode {code}:")
    print("Market and Exchange Names:")
    for n in names:
        print(f"  {n}")
    print("Commodity Names:")
    for c in commodity_names:
        print(f"  {c}") 
    print("Contract Market Names:")
    for cm in contract_market_names:
        print(f"  {cm}")


Code 067651:
Market and Exchange Names:
  CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE
  WTI-PHYSICAL - NEW YORK MERCANTILE EXCHANGE
Commodity Names:
  CRUDE OIL
Contract Market Names:
  WTI-PHYSICAL

Code 067411:
Market and Exchange Names:
  CRUDE OIL, LIGHT SWEET - ICE EUROPE
  CRUDE OIL, LIGHT SWEET - ICE FUTURES EUROPE
  CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE
Commodity Names:
  CRUDE OIL
Contract Market Names:
  CRUDE OIL, LIGHT SWEET-WTI


In [6]:
# Filter: keep only WTI (067651) and ICE Brent (067411)
wti_codes = ["067651", "067411"]
wti_disagg = disagg[disagg["cftc_contract_market_code"].isin(wti_codes)].copy()

# Parse date and coerce the columns we need
wti_disagg["date"] = pd.to_datetime(wti_disagg["report_date_as_yyyy_mm_dd"])
wti_disagg["prod_merc_positions_long"] = pd.to_numeric(wti_disagg["prod_merc_positions_long"], errors="coerce")
wti_disagg["prod_merc_positions_short"] = pd.to_numeric(wti_disagg["prod_merc_positions_short"], errors="coerce")

# Sum NYMEX + ICE per date
commercial = (
    wti_disagg.groupby("date")[["prod_merc_positions_long", "prod_merc_positions_short"]]
    .sum()
    .reset_index()
)
commercial.columns = ["date", "CommercialLong", "CommercialShort"]

print(f"Commercial (disagg PM, NYMEX+ICE): {len(commercial)} dates")
commercial.head()

Commercial (disagg PM, NYMEX+ICE): 1033 dates


,date,CommercialLong,CommercialShort
0,2006-06-13,363858,455968
1,2006-06-20,325138,425078
2,2006-06-27,325150,423772
3,2006-07-03,327513,438406
4,2006-07-11,348625,452829


## Step 2 — Filter CFTC legacy for WTI contracts

Bloomberg's "ManagedMoney" = Legacy Non-Commercial (NOT disaggregated Managed Money).

In [7]:
# Filter legacy for the same two codes
wti_legacy = legacy[legacy["cftc_contract_market_code"].isin(wti_codes)].copy()

wti_legacy["date"] = pd.to_datetime(wti_legacy["report_date_as_yyyy_mm_dd"])
wti_legacy["noncomm_positions_long_all"] = pd.to_numeric(wti_legacy["noncomm_positions_long_all"], errors="coerce")
wti_legacy["noncomm_positions_short_all"] = pd.to_numeric(wti_legacy["noncomm_positions_short_all"], errors="coerce")

# Sum NYMEX + ICE per date
managed_money = (
    wti_legacy.groupby("date")[["noncomm_positions_long_all", "noncomm_positions_short_all"]]
    .sum()
    .reset_index()
)
managed_money.columns = ["date", "MMLong", "MMShort"]

print(f"ManagedMoney (legacy NonComm, NYMEX+ICE): {len(managed_money)} dates")
managed_money.head()

ManagedMoney (legacy NonComm, NYMEX+ICE): 1601 dates


,date,MMLong,MMShort
0,1995-03-21,41647,5014
1,1995-03-28,66095,2867
2,1995-04-04,71734,5578
3,1995-04-18,67110,4143
4,1995-04-25,69552,3541


## Step 3 — Merge and compute net positions

In [8]:
rebuilt = pd.merge(commercial, managed_money, on="date")
rebuilt["CommercialNet"] = rebuilt["CommercialLong"] - rebuilt["CommercialShort"]
rebuilt["MMNet"] = rebuilt["MMLong"] - rebuilt["MMShort"]

print(f"Rebuilt CL: {len(rebuilt)} rows")
rebuilt.head()

Rebuilt CL: 1033 rows


,date,CommercialLong,CommercialShort,MMLong,MMShort,CommercialNet,MMNet
0,2006-06-13,363858,455968,175213,122434,-92110,52779
1,2006-06-20,325138,425078,165196,120527,-99940,44669
2,2006-06-27,325150,423772,178071,111522,-98622,66549
3,2006-07-03,327513,438406,191304,103920,-110893,87384
4,2006-07-11,348625,452829,205885,111565,-104204,94320


## Step 4 — Compare against Marouen's cot_db

In [9]:
# Get Marouen's CL rows (only rows with data)
marouen["tradeDate"] = pd.to_datetime(marouen["tradeDate"])
cl = marouen[marouen["Name"] == "CL"].dropna(subset=["Commercial_NetPosition"]).copy()

# Merge on date
comp = pd.merge(
    rebuilt, cl,
    left_on="date", right_on="tradeDate",
    how="inner",
)
print(f"Matched rows: {len(comp)} (Marouen has {len(cl)})\n")

# Compare each column — relative error (MAPE) + max absolute diff
pairs = [
    ("CommercialLong",  "CommercialLongPosition"),
    ("CommercialShort", "CommercialShortPosition"),
    ("CommercialNet",   "Commercial_NetPosition"),
    ("MMLong",          "ManagedMoney_LongPosition"),
    ("MMShort",         "ManagedMoney_ShortPosition"),
    ("MMNet",           "ManagedMoney_NetPosition"),
]

print(f"{'rebuilt':20s}   {'marouen':30s}   {'MAPE':>8s}   {'MAE':>8s}   {'max_diff':>8s}")
print("-" * 90)
for ours, theirs in pairs:
    diff = (comp[ours] - comp[theirs])
    abs_diff = diff.abs()
    mae = abs_diff.mean()
    denom = comp[theirs].abs().replace(0, float("nan"))
    mape = (abs_diff / denom * 100).mean()
    print(f"{ours:20s}   {theirs:30s}   {mape:7.4f}%   {mae:8.2f}   {abs_diff.max():.0f}")

Matched rows: 806 (Marouen has 813)

rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition            0.0000%       0.00   0
CommercialShort        CommercialShortPosition           0.0000%       0.00   0
CommercialNet          Commercial_NetPosition            0.0000%       0.00   0
MMLong                 ManagedMoney_LongPosition         0.0001%       0.43   2
MMShort                ManagedMoney_ShortPosition        0.0003%       0.44   2
MMNet                  ManagedMoney_NetPosition          0.0002%       0.74   4


In [10]:
# Show a few rows side by side
comp[["date",
      "CommercialLong", "CommercialLongPosition",
      "CommercialShort", "CommercialShortPosition",
      "MMLong", "ManagedMoney_LongPosition",
      "MMShort", "ManagedMoney_ShortPosition",
]].tail(10)

,date,CommercialLong,CommercialLongPosition,CommercialShort,CommercialShortPosition,MMLong,ManagedMoney_LongPosition,MMShort,ManagedMoney_ShortPosition
796,2025-05-27,1084940,1084940.0,757802,757802.0,436187,436187.0,210054,210056.0
797,2025-06-03,1093076,1093076.0,801976,801976.0,466706,466706.0,213514,213514.0
798,2025-06-10,1110267,1110267.0,837322,837322.0,473155,473155.0,193540,193539.0
799,2025-06-17,1177606,1177606.0,858659,858659.0,476885,476885.0,186633,186633.0
800,2025-06-24,1121249,1121249.0,799857,799857.0,468016,468016.0,169501,169502.0
801,2025-07-01,1117041,1117041.0,805503,805503.0,462605,462605.0,158045,158046.0
802,2025-07-08,1154343,1154343.0,817526,817526.0,433121,433121.0,168284,168285.0
803,2025-07-15,1226014,1226014.0,857326,857326.0,404769,404769.0,186816,186815.0
804,2025-07-22,1179861,1179861.0,824552,824552.0,406472,406472.0,197636,197636.0
805,2025-07-29,1172427,1172427.0,809991,809991.0,411275,411275.0,203933,203933.0
